# 02 — Repetições e separação de treino e teste

Vamos preparar uma divisão reproduzível da base, reservando **20% para teste** e **80% para treino**, conforme o roteiro. A análise detalhada e as decisões sobre transformações serão orientadas apenas pelo treino.

**Treino:** dados usados para estudar padrões e desenvolver os modelos, com validação interna para selecionar configurações. **Teste:** conjunto reservado para a avaliação final. **Estratificação:** divisão que mantém aproximadamente a proporção das classes em cada conjunto.

Este notebook é independente do primeiro: execute todas as células em ordem com o kernel da `.venv` e o CSV original disponível conforme o README.


## 1. Localizar e ler o original

A leitura usa uma cópia em memória. O SHA-256 permite conferir a integridade do arquivo antes e depois desta etapa.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from IPython.display import display

relativo = Path("data/raw/credit_risk_dataset.csv")

raiz = None
for p in (Path.cwd(), *Path.cwd().parents):
	if (p / relativo).is_file():
		raiz = p
		break

if raiz is None:
    raise FileNotFoundError("Salve a base original em data/raw/credit_risk_dataset.csv, conforme o README.")

caminho_original = raiz / relativo
sha_original = hashlib.sha256(caminho_original.read_bytes()).hexdigest()
assert sha_original == "ce3c6d2167717bf1627d1c0c81cbccd28323cd4aa7b96d542599366d5ff6aac8", "Base diferente da versão registrada."
df_original = pd.read_csv(caminho_original)
print(f"Base original: {len(df_original):,} registros e {df_original.shape[1]} colunas.")


Base original: 32,581 registros e 12 colunas.


## 2. Decidir sobre as linhas repetidas

A base não possui identificador de contrato ou cliente. Portanto, não podemos provar que duas linhas iguais sejam o mesmo empréstimo. Para este projeto, adotamos a hipótese de trabalho de que **linhas idênticas em todas as 12 colunas são repetições**, mantendo a primeira ocorrência.

Isso atende ao requisito de remoção de duplicatas, evita dar peso extra a registros repetidos e impede que cópias idênticas sejam distribuídas entre treino e teste, o que tornaria a avaliação excessivamente otimista. A limitação é que empréstimos realmente distintos, mas idênticos nas colunas disponíveis, também podem ser removidos.

Essa regra de igualdade é aplicada antes da divisão e não estima média, mediana, escala ou qualquer parâmetro estatístico. Não removemos pessoas por terem apenas idade ou renda iguais, nem filtramos resultados favoráveis ao modelo.


In [2]:
duplicatas = int(df_original.duplicated().sum())
df_base = df_original.drop_duplicates(keep="first").copy()
assert not df_base.duplicated().any()
assert len(df_original) - len(df_base) == duplicatas
print(f"Repetições removidas da cópia em memória: {duplicatas}")
print(f"Registros disponíveis para divisão: {len(df_base):,}")


Repetições removidas da cópia em memória: 165
Registros disponíveis para divisão: 32,416


## 3. Separar atributos e alvo

`X` contém as informações de entrada. `y` contém o resultado que queremos prever: `loan_status`. O alvo não pode ser uma entrada do modelo.

Também verificamos se há atributos idênticos com alvos diferentes. Se surgirem em uma nova base, esta verificação interromperá o fluxo para investigar e planejar uma divisão por grupos, evitando registros de entrada idênticos nos dois conjuntos.


In [3]:
ALVO = "loan_status"
X = df_base.drop(columns=[ALVO])
y = df_base[ALVO]
assert ALVO not in X.columns
assert y.notna().all() and set(y.unique()) == {0, 1}
assert not X.duplicated().any(), "Há entradas idênticas com alvos diferentes; investigar antes da divisão."
print(f"Atributos: {X.shape[1]}")
print(f"Alvo: {y.name}")


Atributos: 11
Alvo: loan_status


## 4. Fazer a divisão estratificada

`test_size=0.20` reserva 20% dos registros. `stratify=y` preserva aproximadamente a proporção de adimplentes e inadimplentes. `random_state=42` fixa a semente do sorteio, permitindo repetir a mesma divisão com a mesma base, ordem e ambiente.

Nenhuma imputação, padronização, codificação ou reamostragem foi aplicada. O balanceamento e o ajuste das transformações serão feitos exclusivamente no treino, dentro do pipeline e das partições de validação.


In [4]:
TEST_SIZE = 0.20
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"Treino: {len(X_train):,} registros")
print(f"Teste: {len(X_test):,} registros")


Treino: 25,932 registros
Teste: 6,484 registros


## 5. Conferir a divisão

Os índices ainda correspondem às posições das linhas na leitura original. Verificamos que nenhuma linha participa dos dois conjuntos, que nenhuma linha da base deduplicada se perdeu e que `X` e `y` permanecem alinhados.

A comparação de proporções abaixo verifica apenas a estratificação; ela não será usada para escolher tratamentos ou modelos a partir do teste. Pequenas diferenças são esperadas porque os tamanhos das classes são números inteiros.


In [5]:
assert X_train.index.intersection(X_test.index).empty
assert set(X_train.index) | set(X_test.index) == set(df_base.index)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.merge(X_test, how="inner", on=list(X.columns)).empty

resumo_classes = pd.DataFrame({
    "base_sem_repeticoes_n": y.value_counts().sort_index(),
    "treino_n": y_train.value_counts().sort_index(),
    "teste_n": y_test.value_counts().sort_index(),
    "base_sem_repeticoes_pct": y.value_counts(normalize=True).sort_index() * 100,
    "treino_pct": y_train.value_counts(normalize=True).sort_index() * 100,
    "teste_pct": y_test.value_counts(normalize=True).sort_index() * 100,
}).rename_axis(ALVO)
display(resumo_classes.round(2))
print("Conjuntos sem sobreposição, completos e alinhados.")


,base_sem_repeticoes_n,treino_n,teste_n,base_sem_repeticoes_pct,treino_pct,teste_pct
loan_status,,,,,,
0,25327,20261,5066,78.13,78.13,78.13
1,7089,5671,1418,21.87,21.87,21.87


Conjuntos sem sobreposição, completos e alinhados.


## 6. Salvar os conjuntos para as próximas etapas

Salvaremos `treino.csv` e `teste.csv` em `data/processed/`, cada um com os 11 atributos e o alvo. O índice temporário do pandas não será exportado como uma coluna de entrada.

Os arquivos ainda contêm nulos e valores discrepantes: **separar os dados não significa concluir a limpeza**. A nova variável obrigatória também será criada posteriormente. O conjunto de teste deve permanecer reservado até a avaliação final.

`separacao.json` registra a base de origem, a regra de repetição, a semente, as versões e os hashes dos arquivos gerados. Os três arquivos ficam locais conforme o `.gitignore`. Reexecutar este notebook recria os mesmos conjuntos.


In [6]:
pasta_saida = raiz / "data/processed"
pasta_saida.mkdir(parents=True, exist_ok=True)
treino = pd.concat([X_train, y_train], axis=1).loc[:, df_base.columns]
teste = pd.concat([X_test, y_test], axis=1).loc[:, df_base.columns]
caminho_treino = pasta_saida / "treino.csv"
caminho_teste = pasta_saida / "teste.csv"
treino.to_csv(caminho_treino, index=False)
teste.to_csv(caminho_teste, index=False)

metadados = {
    "origem": "data/raw/credit_risk_dataset.csv",
    "sha256_original": sha_original,
    "regra_duplicatas": "Igualdade em todas as colunas; manter primeira ocorrência.",
    "registros_originais": len(df_original),
    "repeticoes_removidas": duplicatas,
    "registros_sem_repeticoes": len(df_base),
    "alvo": ALVO,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratify": ALVO,
    "registros_treino": len(treino),
    "registros_teste": len(teste),
    "versoes": {"python": platform.python_version(), "pandas": pd.__version__, "scikit-learn": sklearn.__version__},
    "sha256_treino": hashlib.sha256(caminho_treino.read_bytes()).hexdigest(),
    "sha256_teste": hashlib.sha256(caminho_teste.read_bytes()).hexdigest(),
}
(pasta_saida / "separacao.json").write_text(json.dumps(metadados, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Arquivos gerados: data/processed/treino.csv, teste.csv e separacao.json")


Arquivos gerados: data/processed/treino.csv, teste.csv e separacao.json


## 7. Verificar os arquivos gravados e a preservação do original

A releitura confere se as tabelas exportadas preservaram os dados e os tipos. Pequenas diferenças de representação decimal no CSV são aceitas até `1e-12`; elas não são imputação ou escalonamento.


In [7]:
treino_lido = pd.read_csv(caminho_treino)
teste_lido = pd.read_csv(caminho_teste)
pd.testing.assert_frame_equal(treino.reset_index(drop=True), treino_lido, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(teste.reset_index(drop=True), teste_lido, check_exact=False, rtol=1e-12, atol=1e-12)
assert not treino_lido.drop(columns=ALVO).merge(teste_lido.drop(columns=ALVO), how="inner").shape[0]
assert hashlib.sha256(caminho_original.read_bytes()).hexdigest() == sha_original
assert treino_lido.isna().sum().sum() + teste_lido.isna().sum().sum() == df_base.isna().sum().sum()
print("Exportação conferida; nulos preservados e arquivo original intacto.")


Exportação conferida; nulos preservados e arquivo original intacto.


## Resultado e próxima etapa

A divisão foi feita após a remoção justificada das repetições exatas. O teste permanece reservado; nenhuma transformação estatística foi ajustada.

O próximo notebook utilizará **apenas `data/processed/treino.csv`** para produzir as estatísticas descritivas, investigar nulos e valores discrepantes e construir os gráficos da análise exploratória. A escolha dos hiperparâmetros e do ponto de corte usará validação interna ao treino.
